## 1. Import the libraries:

In [1]:
import pandas as pd
import os

In [2]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model\\research'

In [4]:
# os.chdir("../")
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

# 2. Import the training data:

In [5]:
train = pd.read_parquet("Artifacts/data_cleaning/data_clipped/train.parquet")

In [10]:
train.head(3)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,orders_returned,orders_cancelled,quantity_abs,quantity_abs_clipped,price_clipped
0,546782,22170,PICTURE FRAME WOOD TRIPLE PORTRAIT,4,2011-03-17 09:13:00,6.75,15291.0,United Kingdom,0,0,4,4,6.75
1,535748,21361,LOVE LARGE WOOD LETTERS,1,2010-11-28 13:21:00,12.75,14508.0,United Kingdom,0,0,1,1,12.75
2,490917,21527,RETRO SPOT TRADITIONAL TEAPOT,4,2009-12-08 13:14:00,7.95,15162.0,United Kingdom,0,0,4,4,7.95


# 3. Define the churn label:

- The transactions after the date `2011-03-31` are already dropped during data ingestion.
- The observation window would be `2009-12-01`(this is the first purchase invoice date recorded in the dataset) till `2010-12-31` + filter the customer out if the purchase quantity is just 1
- The churn window would be **3 months + the observation window** i.e. `2011-01-01` to `2011-03-31`

#### Logic implemented for defining the churn label:
- The churn window will be used as a filter to define the churn(1) or retained(0)
- If customer didn't make a single purchase in **the churn window** then 1(churn) else 0(retained)

#### Steps involved:
1. The `dataframe` will be split into two dataframes: `df with observation window(observation_df)` and `df with churn window(churn_df)`
2. Get the unique `customerid` from `churn_df`
3. Create a new feature `churn` on `observation_df`, if `customerid` is in `customerid` of `churn_df` then 0 (retained) else 1(churn).

In [37]:
observation_df = train[train['invoicedate'] < '2011-01-01']
churn_df = train[(train['invoicedate'] >= '2011-01-01') & (train['invoicedate'] <= '2011-03-21')] 

In [38]:
observation_df.shape

(341428, 13)

In [17]:
observation_df.shape

(341428, 13)

In [19]:
churn_df.shape

(46657, 13)

In [29]:
print(f"The min invoice date in observation df: {observation_df['invoicedate'].min()} and the max is: {observation_df['invoicedate'].max()}")
print(f"{'-' * 25}")

customer_ids_from_observation_df = set(observation_df['customerid'].unique())
print(f"There are a total of: {len(customer_ids_from_observation_df)} customer IDs who made the transaction in observation window.")

The min invoice date in observation df: 2009-12-01 07:45:00 and the max is: 2010-12-23 16:06:00
-------------------------
There are a total of: 4379 customer IDs who made the transaction in observation window.


In [26]:
print(f"The min invoice date in churn df: {churn_df['invoicedate'].min()} and the max is: {churn_df['invoicedate'].max()}")

The min invoice date in churn df: 2011-01-04 10:00:00 and the max is: 2011-03-20 16:12:00


In [28]:
customer_ids_from_churn_df = set(churn_df['customerid'].unique())
print(f"There are a total of: {len(customer_ids_from_churn_df)} customer IDs who made the transaction in churn window.")

There are a total of: 1608 customer IDs who made the transaction in churn window.


In [34]:
test = (~observation_df['customerid'].isin(customer_ids_from_churn_df)).astype(int)

In [39]:
test.value_counts()

customerid
0    194117
1    147311
Name: count, dtype: int64

In [41]:
341428

341428

In [42]:
194117+147311

341428

# 4. Start Creating New Features:

### 4.1 Adding `total_price` column: